# Audio Fine-tuning Walkthrough; LFM2.5-Audio-1.5B on Jenny TTS

> 📓 **For learning, not submission.** This walkthrough runs the audio fine-tune cell-by-cell so you can inspect intermediate state. For an **actual demo-day submission**, use `./scripts/audio/launch_hf_job.sh` → `scripts/audio/train.py` directly; that's the same flow as this notebook, end-to-end on HF Jobs without the notebook overhead.

By the end you will have:

1. A fine-tuned LFM2.5-Audio-1.5B that speaks in the Irish female voice from the Jenny TTS dataset
2. The model pushed to your HuggingFace Hub account
3. A working TTS inference snippet you can drop into your demo

**Time:** about 20 minutes if you're running on Colab/Jupyter with a T4 (smoke-sized training) or about 60
minutes for the full 5000-step recipe on an A100 80GB.

**Cost on HF Jobs:** about $2.50 (one A100-hour) for the full recipe.

**Prereqs:** HuggingFace account with your $150 hackathon credit applied
(the claim link is shared with registered teams), W&B account.

## 0. Setup

Install `liquid-audio` (pinned to a torchcodec-free main-branch SHA, so no system FFmpeg is needed
anywhere), plus `wandb` so the trainer can stream scalars and `python-dotenv` for local credential
loading.

In [ ]:
%pip install -q "liquid-audio @ git+https://github.com/Liquid4All/liquid-audio@84c173b2208271dec130d0af2cfd7333a09433e1" wandb python-dotenv

Set your HuggingFace and W&B credentials. On Colab you can use `userdata`; locally, drop them in a `.env`.

In [ ]:
import os

IS_COLAB = "COLAB_" in "".join(os.environ.keys())
if IS_COLAB:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    os.environ["WANDB_ENTITY"] = userdata.get("WANDB_ENTITY")
    os.environ["WANDB_PROJECT"] = userdata.get("WANDB_PROJECT")
else:
    from dotenv import load_dotenv

    load_dotenv()

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN before continuing"
print("credentials loaded")

## 1. Concepts: how LFM2-Audio sees a dataset

LFM2-Audio is a chat model; it speaks in **turns** (`system`, `user`, `assistant`) like any LLM, except
that each turn can contain text *or* audio *or* both. The data primitives are:

| Type | Use |
| --- | --- |
| `TextSegment(text="...")` | A run of text tokens |
| `AudioSegment(audio=<bytes or tensor>)` | A run of audio. On the way in, gets log-mel features. On the way out, gets Mimi codes. |
| `InterleavedSegment(text=..., audio=...)` | A run where text and audio tokens are interleaved at the model's fixed cadence (used for real-time speech-to-speech). |
| `ChatMessage(role="...", content=[<segments>])` | One turn |

For TTS finetuning, every training sample is a three-turn chat:

```
[
  ChatMessage(role="system",     content=[TextSegment("Perform TTS. Use the Irish female voice.")]),
  ChatMessage(role="user",       content=[TextSegment("<the line to read>")]),
  ChatMessage(role="assistant",  content=[AudioSegment(<the recording>)]),
]
```

The model already speaks four built-in voices (US M/F, UK M/F). The fine-tune teaches it a fifth voice
(Irish female, from Jenny). The system prompt is what triggers the voice; choose a unique one for your
voice so the new voice doesn't collide with the built-ins.

Under the hood, audio is tokenized into **8 Mimi codebooks** at 12.5 Hz. Each "audio token" the model
emits is actually 8 integers (one per codebook). Mimi decodes the 8-stream sequence back to 24 kHz
waveform.

## 2. Peek at the Jenny TTS dataset

The reference dataset is `reach-vb/jenny_tts_dataset`; single Irish female voice, around 30 hours, around
20,000 samples. Each row has a `transcription` (text) and an `audio` field (the recording).

In [ ]:
from datasets import Audio, load_dataset

ds = load_dataset("reach-vb/jenny_tts_dataset", split="train", streaming=True)
ds = ds.cast_column("audio", Audio(decode=False))  # we pass raw bytes, mapper handles decoding

row = next(iter(ds))
print(f"transcription: {row['transcription']!r}")
print(f"audio bytes:   {len(row['audio']['bytes']):,} bytes")
print(f"audio path:    {row['audio']['path']!r}")

## 3. Preprocess: dataset -> on-disk shards

`LFM2AudioChatMapper` does the heavy lifting (text tokenization, log-mel features for input audio, Mimi
tokenization for output audio). `preprocess_dataset` orchestrates: iterates your data, applies the
mapper, writes shards to disk.

Two important knobs:

- `max_context_length`: hard cap on tokens per sample. Default 256. Around 163 of the 20K Jenny samples
  exceed this and get skipped; that's expected (matches the upstream recipe).
- `output_path`: where the preprocessed shards land. The trainer reads from here.

Preprocessing runs Mimi encoding on GPU, so it benefits from a GPU. On a T4 expect about 10 minutes;
on an H100, around 7 minutes for the full dataset.

In [ ]:
from liquid_audio import LFM2AudioProcessor
from liquid_audio.data.mapper import LFM2AudioChatMapper
from liquid_audio.data.preprocess import preprocess_dataset
from liquid_audio.data.types import AudioSegment, ChatMessage, TextSegment


class TrainingSamples:
    """Yields one `list[ChatMessage]` per training sample.

    This class IS the dataset interface; rewrite `__iter__` for your own
    data and nothing else changes. For TTS each sample is three turns:
    the system voice prompt, the text to speak, the target audio as bytes.

    EXAMPLE: streams the Jenny TTS dataset. `limit` keeps walkthrough
    iteration tight; pass `limit=0` for the full recipe.

    (A class rather than a bare generator function because the upstream
    preprocessor pickles the data source for dataset fingerprinting, and
    generator objects can't be pickled.)
    """

    def __init__(self, limit: int = 200):
        self.limit = limit

    def __iter__(self):
        ds = load_dataset("reach-vb/jenny_tts_dataset", split="train", streaming=True)
        ds = ds.cast_column("audio", Audio(decode=False))  # keep raw bytes; liquid-audio decodes
        for i, row in enumerate(ds):
            if self.limit and i >= self.limit:
                break
            yield [
                ChatMessage(role="system", content=[TextSegment(text="Perform TTS. Use the Irish female voice.")]),
                ChatMessage(role="user", content=[TextSegment(text=row["transcription"])]),
                ChatMessage(role="assistant", content=[AudioSegment(audio=row["audio"]["bytes"])]),
            ]

In [ ]:
processor = LFM2AudioProcessor.from_pretrained("LiquidAI/LFM2.5-Audio-1.5B", device="cuda").eval()
mapper = LFM2AudioChatMapper(processor)

preprocess_dataset(
    data=TrainingSamples(limit=200),
    output_path="data/jenny_tts/train_small",
    mapper=mapper,
    max_context_length=256,
)

## 4. Train

`liquid_audio.trainer.Trainer` is `accelerate`-backed. The full upstream recipe is **bs=64, ctx=256,
lr=1e-4, max-steps=5000, warmup=250**, which takes around 50 minutes on 1× A100 80GB. For the notebook we
use a much smaller `max_steps` so you can see the loop end-to-end in a few minutes.

One catch: the upstream `Trainer` only **prints** metrics to stdout; it never calls `wandb.log()`, so
setting `WANDB_API_KEY` alone gets you nothing. The tiny `WandbTrainer` subclass below forwards loss + lr
to W&B (it's the same subclass `scripts/audio/train.py` uses).

In [ ]:
import os

import wandb
from liquid_audio.data.dataloader import LFM2DataLoader
from liquid_audio.trainer import Trainer


# Same WandbTrainer as the production scripts/audio/train.py, inlined so this walkthrough is self-contained; keep them in sync.
class WandbTrainer(Trainer):
    """Upstream Trainer prints metrics but never calls wandb.log(); forward them."""

    def log(self, model_output):
        super().log(model_output)
        if self.step > 0 and self.step % self.logging_interval == 0:
            wandb.log(
                {"train/loss": model_output.loss.item(), "train/lr": self.optimizer.param_groups[0]["lr"]},
                step=self.step,
            )


wandb.init(project=os.environ.get("WANDB_PROJECT", "hack-the-liquid-way"), tags=["audio", "walkthrough"])

trainer = WandbTrainer(
    model_id="LiquidAI/LFM2.5-Audio-1.5B",
    train_data=LFM2DataLoader(dataset_path="data/jenny_tts/train_small", context_length=256),
    lr=1e-4,
    batch_size=8,  # walkthrough size; upstream uses 64
    max_steps=100,  # walkthrough size; upstream uses 5000
    warmup_steps=10,  # upstream uses 250
    dataloader_num_workers=4,
    logging_interval=10,
    save_interval=50,
    output_dir="ckpt-walkthrough",
)
trainer.train()
wandb.finish()

Loss should descend monotonically from about 2.1 to about 2.05 over 100 steps on the small slice. The
absolute number isn't directly comparable to LM perplexity; it's a sum over 8 Mimi codebook heads + a
text head; so focus on the trajectory, not the value.

Once it's done, your checkpoint lives at `ckpt-walkthrough/`. Time to test it.

## 5. Inference: hear the fine-tune speak

Load the checkpoint exactly like you would the base model; same classes, just a local path instead of
an HF repo id. Then build a `ChatState`, set the same system prompt the model was fine-tuned on, give
it a sentence to read, and stream tokens out of `generate_sequential`.

In [ ]:
import soundfile as sf
import torch
from IPython.display import Audio, display
from liquid_audio import ChatState, LFM2AudioModel

CHECKPOINT = "ckpt-walkthrough"

ft_processor = LFM2AudioProcessor.from_pretrained(CHECKPOINT).eval()
ft_model = LFM2AudioModel.from_pretrained(CHECKPOINT).eval().to("cuda")

chat = ChatState(ft_processor)
chat.new_turn("system")
chat.add_text("Perform TTS. Use the Irish female voice.")
chat.end_turn()

chat.new_turn("user")
chat.add_text("Welcome to Hack the Liquid WAY. The Irish voice is alive.")
chat.end_turn()

chat.new_turn("assistant")

audio_out: list[torch.Tensor] = []
for t in ft_model.generate_sequential(
    **chat, max_new_tokens=512, audio_temperature=0.8, audio_top_k=64
):
    if t.numel() > 1:
        audio_out.append(t)

# Drop the trailing end-of-audio frame, then run Mimi decode.
audio_codes = torch.stack(audio_out[:-1], 1).unsqueeze(0)
waveform = ft_processor.decode(audio_codes)
sf.write("hello_irish.wav", waveform.cpu().squeeze().numpy(), 24_000)
display(Audio("hello_irish.wav"))

100 steps is far from convergence; the voice will sound *closer* to Jenny than the base model's voices,
but it'll be noisy. To get production quality, push past 1000 steps (about 10 minutes on an A100).

## 6. Push the fine-tune to HuggingFace Hub

So your teammates and the judges can load the model with one line, push to your account:

In [ ]:
from huggingface_hub import HfApi

REPO_ID = "<your-username>/lfm25-audio-jenny-walkthrough"  # change this

api = HfApi()
api.create_repo(REPO_ID, private=True, exist_ok=True)
api.upload_folder(
    folder_path=CHECKPOINT,
    repo_id=REPO_ID,
    commit_message="walkthrough checkpoint",
)
print(f"https://huggingface.co/{REPO_ID}")

After this, anyone (with read access if it's private) can pull the model:

```python
ft_processor = LFM2AudioProcessor.from_pretrained("<your-username>/lfm25-audio-jenny-walkthrough")
ft_model     = LFM2AudioModel.from_pretrained("<your-username>/lfm25-audio-jenny-walkthrough")
```

## 7. Next steps for the hackathon

1. **Custom dataset**: rewrite `TrainingSamples.__iter__` to yield `list[ChatMessage]` from your own data.
   The three-turn structure stays the same. For Japanese TTS, source an open-license JP corpus (e.g. `reazon-research/reazonspeech`) and swap the system prompt accordingly (`Perform TTS. Use the Japanese male voice.`).
2. **Scale up**: pass `limit=0`, bump `max_steps=5000` and `batch_size=64`, submit via
   `./scripts/audio/launch_hf_job.sh` instead of running locally. About $2.50 / one A100-hour.
3. **Different tasks**: this notebook covers single-turn, single-voice TTS. The same `ChatMessage`
   plumbing supports any sequence of (text/audio)-to-(text/audio), mixed in arbitrary ways. Be
   creative: voice editing, speech classification, diarization, meeting summarization, and more.
4. **On-device demo**: see the on-device section of `examples/audio/README.md` for the AMD Ryzen AI
   PC path (remember to pre-record a video demo to complement your live
   demo).

When you're stuck, ping `#hack-the-liquid-way` on Discord. Happy fine-tuning.